# HR Analytics — Feature Engineering Notebook
Engagement Index · Burnout Score · Workload Stress · Stability Score

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
warnings_module = __import__('warnings')
warnings_module.filterwarnings('ignore')

df_raw = pd.read_csv('../data/raw/HR_Employee_Attrition.csv')
print('Raw shape:', df_raw.shape)

In [ ]:
# === ENGAGEMENT INDEX ===
components = ['JobInvolvement','JobSatisfaction','EnvironmentSatisfaction','RelationshipSatisfaction']
weights = np.array([0.30, 0.30, 0.20, 0.20])

scaler = MinMaxScaler()
scaled = scaler.fit_transform(df_raw[components])
df_raw['EngagementIndex'] = scaled @ weights
df_raw['EngagementBand'] = pd.cut(df_raw['EngagementIndex'], bins=[-0.001,0.33,0.66,1.001], labels=['Low','Medium','High'])

print('Engagement Index Stats:')
print(df_raw['EngagementIndex'].describe())
print('\nBand Distribution:')
print(df_raw['EngagementBand'].value_counts())

In [ ]:
# Visualize Engagement Index
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].hist(df_raw['EngagementIndex'], bins=30, color='#667eea', edgecolor='white')
axes[0].set_title('Engagement Index Distribution')
axes[0].axvline(df_raw['EngagementIndex'].mean(), color='red', linestyle='--', label='Mean')
axes[0].legend()

df_raw['EngagementBand'].value_counts().plot(kind='bar', ax=axes[1], color=['#ef4444','#f59e0b','#10b981'], rot=0)
axes[1].set_title('Engagement Band Counts')

for attrition_val, color in [('No','#10b981'),('Yes','#ef4444')]:
    subset = df_raw[df_raw['Attrition']==attrition_val]['EngagementIndex']
    axes[2].hist(subset, bins=25, alpha=0.7, label=f'Attrition={attrition_val}', color=color)
axes[2].set_title('Engagement Index by Attrition')
axes[2].legend()
plt.tight_layout()
plt.show()

In [ ]:
# === BURNOUT RISK SCORE ===
score = pd.Series(0.0, index=df_raw.index)
df_raw['OT_Enc'] = df_raw['OverTime'].map({'Yes':1,'No':0})
score += df_raw['OT_Enc'] * 0.35
score += (4 - df_raw['WorkLifeBalance'].clip(1,4)) / 3.0 * 0.30
bt_map = {'Non-Travel':0,'Travel_Rarely':0.5,'Travel_Frequently':1.0}
score += df_raw['BusinessTravel'].map(bt_map).fillna(0) * 0.15
sat_avg = df_raw[['JobSatisfaction','EnvironmentSatisfaction','RelationshipSatisfaction']].mean(axis=1)
score += (4 - sat_avg.clip(1,4)) / 3.0 * 0.20
df_raw['BurnoutScore'] = score.clip(0,1)
df_raw['BurnoutRisk'] = pd.cut(df_raw['BurnoutScore'], bins=[-0.001,0.33,0.66,1.001], labels=['Low','Medium','High'])
print('Burnout Risk Distribution:')
print(df_raw['BurnoutRisk'].value_counts())

In [ ]:
# === PCA-BASED ENGAGEMENT (Alternative Method) ===
pca = PCA(n_components=2)
pca_result = pca.fit_transform(scaled)
print(f'PCA Variance Explained: PC1={pca.explained_variance_ratio_[0]:.2%}, PC2={pca.explained_variance_ratio_[1]:.2%}')

fig, ax = plt.subplots(figsize=(8,6))
colors = {'Low':'#ef4444','Medium':'#f59e0b','High':'#10b981'}
for band in ['Low','Medium','High']:
    mask = df_raw['EngagementBand'] == band
    ax.scatter(pca_result[mask,0], pca_result[mask,1], c=colors[band], label=band, alpha=0.6, s=20)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.set_title('PCA of Engagement Components (colored by band)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# === DERIVED FEATURES SUMMARY ===
df_raw['StagnationIndex'] = (df_raw['YearsSinceLastPromotion'] / (df_raw['YearsAtCompany'] + 1)).clip(0,1)
df_raw['CompanyLoyaltyRatio'] = (df_raw['YearsAtCompany'] / (df_raw['TotalWorkingYears'] + 1)).clip(0,1)
df_raw['SatisfactionVariance'] = df_raw[['JobSatisfaction','EnvironmentSatisfaction','RelationshipSatisfaction','WorkLifeBalance']].var(axis=1)
df_raw['SatisfactionStabilityScore'] = 1 - (df_raw['SatisfactionVariance'] / df_raw['SatisfactionVariance'].max())

derived = ['EngagementIndex','BurnoutScore','StagnationIndex','CompanyLoyaltyRatio','SatisfactionStabilityScore']
print('Derived Feature Correlations with Attrition:')
attrition_enc = (df_raw['Attrition'] == 'Yes').astype(int)
for feat in derived:
    corr = df_raw[feat].corr(attrition_enc)
    print(f'  {feat:35s}: {corr:+.4f}')